<a href="https://colab.research.google.com/github/peeush-agarwal/peeush-agarwal.github.io/blob/main/nbs/nlp/04-imdb-reviews-project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IMDB Reviews Dataset for Sentiment Analysis

## Load data

In [10]:
import kagglehub
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews", force_download=True)
path

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.


'/kaggle/input/imdb-dataset-of-50k-movie-reviews'

In [2]:
import pandas as pd

In [11]:
kaggle_path = "/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv"

In [12]:
DATA_FILE = kaggle_path or "IMDB Dataset.csv"

In [13]:
DATA_FILE

'/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv'

In [14]:
df = pd.read_csv(DATA_FILE)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [15]:
df.shape

(50000, 2)

## Split Data into train, validation and test

In [18]:
from sklearn.model_selection import train_test_split


train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)

print(f"Train set: {train_df.shape}")
print(f"Validation set: {val_df.shape}")
print(f"Test set: {test_df.shape}")

Train set: (28000, 2)
Validation set: (7000, 2)
Test set: (15000, 2)


## Text preprocessing

In [23]:
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

stopwords_list = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def clean_text(text):
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = text.lower()
    words = nltk.word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words if word not in stopwords_list]
    return " ".join(words)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [24]:
train_df["review"].iloc[0]

"When I first got my N64 when I was five or six,I fell in love with it,and my first game was Super Mario 64.And I LOVED IT!The graphics were great for it's time,a good plot,great courses and above all,the best music I heard in a Nintendo game.<br /><br />I don't remember the plot completely,but I think Princess Peach was kidnapped by Bowser,and Mario has to rescue her.The object of the game is to get 120 stars from the curses in the castle.Each had about five or six challnges to get the stars.There are secert parts of the castle,where you can get more stars.But of course,you have beat Bowser.*I think there are three levels to beat Bowser on* Lets start with the characters.Mario is the main character,and gets helpful advice from Toad,so he is basically one of your only alliances.I heard that Luigi and Yoshi are in the game towards the end.The main villain is Bowser,and there are a bunch of other characters like Boo and Goomba.The characters are really great.<br /><br />Next,how about th

In [25]:
clean_text(train_df["review"].iloc[0])

'first got n five six fell love first game super mario loved graphic great time good plot great course best music heard nintendo game remember plot completely think princess peach kidnapped bowser mario rescue object game get star curse castle five six challnges get star secert part castle get star course beat bowser think three level beat bowser let start character mario main character get helpful advice toad basically one alliance heard luigi yoshi game towards end main villain bowser bunch character like boo goomba character really great next graphic people say gameplay important graphic agree completely great plot great graphic especially time whole bunch nintendo game like graphic compare super mario bright color great effect awesome sound effect found graphic water course good next bowser world one best graphic game music favorite part game growing played young age gladly leave game night music would put sleep especially music jolly roger bay peaceful wonderful others great espec

In [26]:
train_df["review_cleaned"] = train_df["review"].apply(clean_text)
val_df["review_cleaned"] = val_df["review"].apply(clean_text)
test_df["review_cleaned"] = test_df["review"].apply(clean_text)

## Bag of Words based Feature Extraction

In [27]:
# Bag of Words

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=5000)
X_train_bow = vectorizer.fit_transform(train_df["review_cleaned"]).toarray()
X_val_bow = vectorizer.transform(val_df["review_cleaned"]).toarray()
X_test_bow = vectorizer.transform(test_df["review_cleaned"]).toarray()

In [28]:
y_train = train_df["sentiment"].map({"positive": 1, "negative": 0}).values
y_val = val_df["sentiment"].map({"positive": 1, "negative": 0}).values
y_test = test_df["sentiment"].map({"positive": 1, "negative": 0}).values

In [29]:
print(X_train_bow.shape, y_train.shape)
print(X_val_bow.shape, y_val.shape)
print(X_test_bow.shape, y_test.shape)

(28000, 5000) (28000,)
(7000, 5000) (7000,)
(15000, 5000) (15000,)


### BoW based machine learning model (RandomForest)

In [30]:
# Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier


rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train_bow, y_train)

RandomForestClassifier(random_state=42)

In [31]:
y_val_pred = rf_classifier.predict(X_val_bow)

In [32]:
from sklearn.metrics import accuracy_score, classification_report


accuracy_score(y_val, y_val_pred)

0.8487142857142858

In [33]:
print(classification_report(y_val, y_val_pred))

              precision    recall  f1-score   support

           0       0.84      0.86      0.85      3540
           1       0.86      0.83      0.85      3460

    accuracy                           0.85      7000
   macro avg       0.85      0.85      0.85      7000
weighted avg       0.85      0.85      0.85      7000



In [37]:
y_test_pred = rf_classifier.predict(X_test_bow)
accuracy_score(y_test, y_test_pred)

0.8470666666666666

In [34]:
# Hyperparameter tuning
from sklearn.model_selection import RandomizedSearchCV


param_dist = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

rf_random_search = RandomizedSearchCV(
    estimator=rf_classifier,
    param_distributions=param_dist,
    n_iter=3,
    cv=2,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
)
rf_random_search.fit(X_train_bow, y_train)

Fitting 2 folds for each of 3 candidates, totalling 6 fits


RandomizedSearchCV(cv=2, estimator=RandomForestClassifier(random_state=42),
                   n_iter=3, n_jobs=-1,
                   param_distributions={'max_depth': [None, 10, 20, 30],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [50, 100, 200]},
                   random_state=42, return_train_score=True, verbose=2)

In [35]:
best_rf_model = rf_random_search.best_estimator_
y_val_pred_best = best_rf_model.predict(X_val_bow)
accuracy_score(y_val, y_val_pred_best)

0.8527142857142858

In [36]:
y_test_pred = best_rf_model.predict(X_test_bow)
accuracy_score(y_test, y_test_pred)

0.8482666666666666

## TF-IDF based Feature Extraction

In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer


tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(train_df["review_cleaned"]).toarray()
X_val_tfidf = tfidf_vectorizer.transform(val_df["review_cleaned"]).toarray()
X_test_tfidf = tfidf_vectorizer.transform(test_df["review_cleaned"]).toarray()

In [41]:
print(f"Total words in vocab: {len(tfidf_vectorizer.vocabulary_)}")

Total words in vocab: 5000


### TF-IDF based machine learning model (RandomForest)

In [42]:
rf_classifier_tfidf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier_tfidf.fit(X_train_tfidf, y_train)

RandomForestClassifier(random_state=42)

In [43]:
y_val_pred_tfidf = rf_classifier_tfidf.predict(X_val_tfidf)
accuracy_score(y_val, y_val_pred_tfidf)

0.8544285714285714

In [44]:
y_test_pred_tfidf = rf_classifier_tfidf.predict(X_test_tfidf)
accuracy_score(y_test, y_test_pred_tfidf)

0.8462

## Word2Vec based Feature Extraction

In [46]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 25.6 MB/s eta 0:00:00


In [49]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

In [56]:
sentences_train = train_df["review"].apply(simple_preprocess).tolist()
print(len(sentences_train))

28000


In [57]:
sentences_val = val_df["review"].apply(simple_preprocess).tolist()
sentences_test = test_df["review"].apply(simple_preprocess).tolist()

In [58]:
w2v = Word2Vec(sentences=sentences_train, vector_size=100, window=5, min_count=1, workers=4)

In [61]:
print(len(w2v.wv.key_to_index))

78424


In [63]:
import numpy as np


def get_average_word2vec_vector(words, model, vector_size):
    vector = []
    num_words = 0
    for word in words:
        if word in model.wv:
            vector.append(model.wv[word])
            num_words += 1
    if num_words > 0:
        return sum(vector) / num_words
    else:
        return [0.0] * vector_size


# Get Word2Vec features for training, validation, and test sets
X_train_w2v = [
    get_average_word2vec_vector(words, w2v, w2v.vector_size)
    for words in sentences_train
]
X_val_w2v = [
    get_average_word2vec_vector(words, w2v, w2v.vector_size)
    for words in sentences_val
]
X_test_w2v = [
    get_average_word2vec_vector(words, w2v, w2v.vector_size)
    for words in sentences_test
]


# Convert to numpy arrays
X_train_w2v = np.array(X_train_w2v)
X_val_w2v = np.array(X_val_w2v)
X_test_w2v = np.array(X_test_w2v)

print(f"Shape of X_train_w2v: {X_train_w2v.shape}")
print(f"Shape of X_val_w2v: {X_val_w2v.shape}")
print(f"Shape of X_test_w2v: {X_test_w2v.shape}")


# Train a RandomForest Classifier on Word2Vec features
rf_classifier_w2v = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier_w2v.fit(X_train_w2v, y_train)

# Evaluate on validation set
y_val_pred_w2v = rf_classifier_w2v.predict(X_val_w2v)
accuracy_w2v_val = accuracy_score(y_val, y_val_pred_w2v)

print(f"Word2Vec model accuracy on validation set: {accuracy_w2v_val}")
print("Classification Report on Validation Set (Word2Vec):")
print(classification_report(y_val, y_val_pred_w2v))

Shape of X_train_w2v: (28000, 100)
Shape of X_val_w2v: (7000, 100)
Shape of X_test_w2v: (15000, 100)
Word2Vec model accuracy on validation set: 0.8064285714285714
Classification Report on Validation Set (Word2Vec):
              precision    recall  f1-score   support

           0       0.82      0.80      0.81      3540
           1       0.80      0.82      0.81      3460

    accuracy                           0.81      7000
   macro avg       0.81      0.81      0.81      7000
weighted avg       0.81      0.81      0.81      7000



In [64]:
# Hyperparameter tuning for Word2Vec based Random Forest
from sklearn.model_selection import RandomizedSearchCV

param_dist_w2v = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

rf_random_search_w2v = RandomizedSearchCV(
    estimator=rf_classifier_w2v,  # Use the Word2Vec trained classifier
    param_distributions=param_dist_w2v,
    n_iter=3,  # Number of parameter settings that are sampled
    cv=2,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
)
rf_random_search_w2v.fit(X_train_w2v, y_train)

Fitting 2 folds for each of 3 candidates, totalling 6 fits


RandomizedSearchCV(cv=2, estimator=RandomForestClassifier(random_state=42),
                   n_iter=3, n_jobs=-1,
                   param_distributions={'max_depth': [None, 10, 20, 30],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [50, 100, 200]},
                   random_state=42, return_train_score=True, verbose=2)

In [65]:
best_rf_model_w2v = rf_random_search_w2v.best_estimator_
y_val_pred_best_w2v = best_rf_model_w2v.predict(X_val_w2v)
accuracy_score(y_val, y_val_pred_best_w2v)

0.809

In [66]:
y_test_pred_best_w2v = best_rf_model_w2v.predict(X_test_w2v)
accuracy_score(y_test, y_test_pred_best_w2v)

0.8068666666666666

### SVM with TF-IDF Features

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Initialize and train the SVM classifier
svm_classifier_tfidf = SVC(kernel='linear', random_state=42, verbose=True)
svm_classifier_tfidf.fit(X_train_tfidf, y_train)

[LibSVM]

In [ ]:
# Predict on the validation set
y_val_pred_svm_tfidf = svm_classifier_tfidf.predict(X_val_tfidf)

# Evaluate validation accuracy
accuracy_svm_tfidf_val = accuracy_score(y_val, y_val_pred_svm_tfidf)
print(f"SVM with TF-IDF accuracy on validation set: {accuracy_svm_tfidf_val}")
print("Classification Report on Validation Set (SVM with TF-IDF):")
print(classification_report(y_val, y_val_pred_svm_tfidf))

In [ ]:
# Predict on the test set
y_test_pred_svm_tfidf = svm_classifier_tfidf.predict(X_test_tfidf)

# Evaluate test accuracy
accuracy_svm_tfidf_test = accuracy_score(y_test, y_test_pred_svm_tfidf)
print(f"SVM with TF-IDF accuracy on test set: {accuracy_svm_tfidf_test}")
print("Classification Report on Test Set (SVM with TF-IDF):")
print(classification_report(y_test, y_test_pred_svm_tfidf))